# 05.1 — Tinier-YOLO pe Raspberry Pi

**Țintă:** Raspberry Pi 4, 4 GB, Debian 13 ARM64 (`aarch64`). Urmați
[ghidul instalării](../raspberry_pi/README.md), apoi rulați testul de referință al modelului.

Se folosesc greutățile originale și biblioteca software ARM64 din proiect, fără FPGA.
Aceasta este o referință C++/HLS, nu un motor AI optimizat pentru Raspberry Pi; un apel poate dura minute.
Mesajul periodic indică doar timpul scurs. Întreruperea unei funcții native poate fi întârziată.
**Inferența reală pe Raspberry Pi nu a fost încă validată.**

Notebookul nu salvează automat imagini sau jurnale de rezultate. Jupyter poate salva ieșirile celulelor
în fișierul notebook; ștergeți ieșirile înainte de a salva dacă nu doriți să persiste.


## 1. Mediu

In [ ]:
import sys, gc, time, hashlib, pickle
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from threadpoolctl import threadpool_limits, threadpool_info

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "raspberry_pi/qnn_cpu.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository with raspberry_pi/start_jupyter.sh")
sys.path.insert(0, str(ROOT / "raspberry_pi"))
from qnn_cpu import DorefanetCPU, TinierYoloCPU, DarknetCPU, utils, snapshot, temporary_image, machine_info
if "blas_limit" in globals():
    blas_limit.restore_original_limits()
blas_limit = threadpool_limits(limits=1)
cv2.setNumThreads(1)
QNN_PARAM_DIR = ROOT / "qnn/params"
NOTEBOOK_DIR = ROOT / "notebooks"
print(machine_info())
print(threadpool_info())
BACKGROUND_CAMERA_STREAM = "uStreamer active for Home Assistant"  # record actual condition


## 2. Model și Darknet CPU

In [ ]:
for name in ('classifier', 'darknet'):
    old = globals().get(name)
    if old is not None:
        old.deinit_accelerator() if name == 'classifier' else old.close()
for name in ('conv0_weights', 'conv8_weights', 'conv0_bias', 'conv8_bias'):
    globals().pop(name, None)
gc.collect()
conv0_weights = np.load(QNN_PARAM_DIR / 'tinier-yolo-conv0-W.npy', encoding='latin1', allow_pickle=True).transpose(3,2,1,0)
conv8_weights = np.load(QNN_PARAM_DIR / 'tinier-yolo-conv8-W.npy', encoding='latin1', allow_pickle=True).transpose(3,2,1,0)
conv0_bias = np.load(QNN_PARAM_DIR / 'tinier-yolo-conv0-bias.npy', encoding='latin1', allow_pickle=True).reshape(-1,1)
conv8_bias = np.load(QNN_PARAM_DIR / 'tinier-yolo-conv8-bias.npy', encoding='latin1', allow_pickle=True).reshape(-1,1)
classifier = TinierYoloCPU()
darknet = None
try:
    net = classifier.load_network()
    out_ch, out_dim, _ = net['conv7']['output']
    first_ch, first_dim, _ = net['conv1']['input']
    if conv0_weights.shape[0] != first_ch or conv8_weights.shape[:2] != (125, out_ch):
        raise ValueError('Weights do not match this model.')
    conv0_bias = np.broadcast_to(conv0_bias, (first_ch, first_dim * first_dim))
    conv8_bias = np.broadcast_to(conv8_bias, (125, out_dim * out_dim))
    classifier.init_accelerator()
    darknet = DarknetCPU()
except BaseException:
    classifier.deinit_accelerator()
    if darknet is not None:
        darknet.close()
    raise
print('Model ready.')


## 3. Sursa imaginii

Pentru camera Logitech folosiți `SOURCE = "snapshot"`. uStreamer rămâne singurul proces care deschide camera USB; Home Assistant continuă să primească fluxul. Este procesat un singur cadru, fără buclă video. Pentru comparația PYNQ/CPU folosiți inițial același fișier.

In [ ]:
SOURCE = 'file'  # 'file' or 'snapshot'
IMAGE_NAME = 'dog.jpg'
SNAPSHOT_URL = 'http://127.0.0.1:8080/snapshot'
THRESHOLD = 0.3


## 4. Pipeline CPU

In [ ]:
def process_yolo(image_path, threshold):
    timings = {}
    t0 = time.perf_counter()
    pixels = darknet.load_image(image_path)
    timings['preprocess_s'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    values = utils.conv_layer(pixels, conv0_weights, b=conv0_bias, stride=2, padding=1)
    quantized = utils.quantize(values.clip(0.0, 4.0) / 4.0, 3)
    timings['first_cpu_s'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    output = classifier.get_accel_buffer(out_ch, out_dim)
    inputs = np.ascontiguousarray(classifier.prepare_buffer(quantized * 7))
    timings['packing_s'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    classifier.inference(inputs, output)
    timings['quantized_core_s'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    features = classifier.postprocess_buffer(output)
    timings['unpacking_s'] = time.perf_counter() - t0
    quantized_hash = hashlib.sha256(np.ascontiguousarray(features, dtype='<f4').tobytes()).hexdigest()
    t0 = time.perf_counter()
    pixels = np.swapaxes(features.reshape(out_dim, out_dim, out_ch), 0, 1)[np.newaxis]
    head = np.ascontiguousarray(utils.conv_layer(pixels, conv8_weights, b=conv8_bias, stride=1), dtype=np.float32)
    timings['last_cpu_s'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    preview, detections = darknet.draw(image_path, head, threshold)
    timings['decode_draw_ram_io_s'] = time.perf_counter() - t0
    return preview, detections, timings, quantized_hash


## 5. O inferență

In [ ]:
if SOURCE not in ('file', 'snapshot'):
    raise ValueError("SOURCE must be 'file' or 'snapshot'")
if not 0 < THRESHOLD < 1:
    raise ValueError('THRESHOLD must be between 0 and 1')
capture_seconds = None
if SOURCE == 'file':
    image_path = NOTEBOOK_DIR / 'yoloimages' / IMAGE_NAME
    input_name = image_path.name
    input_sha256 = hashlib.sha256(image_path.read_bytes()).hexdigest()
    with Image.open(image_path) as opened:
        original = opened.convert('RGB')
    preview, detections, timings, quantized_sha256 = process_yolo(image_path, THRESHOLD)
else:
    t0 = time.perf_counter()
    original = snapshot(SNAPSHOT_URL)
    capture_seconds = time.perf_counter() - t0
    with temporary_image(original) as image_path:
        input_name = 'uStreamer snapshot (temporary PNG)'
        input_sha256 = hashlib.sha256(image_path.read_bytes()).hexdigest()
        preview, detections, timings, quantized_sha256 = process_yolo(image_path, THRESHOLD)
print('Input:', input_name, input_sha256)
if capture_seconds is not None:
    print(f'Snapshot acquisition (outside inference): {capture_seconds:.3f} s')
original.thumbnail((640,480))
preview.thumbnail((800,600))
display(original, preview)
for name, score in detections:
    print(f'{name:20s} {score:6.0%}')
if not detections:
    print('No objects above threshold; this can be a valid result.')


### Compararea cu PYNQ
Folosiți aceeași imagine și aceleași greutăți. Comparați `quantized_core_s` cu timpul apelului
`classifier.inference(...)` din notebookul PYNQ. Acesta exclude împachetarea/dezîmpachetarea.
Suma de mai jos include preprocesarea și bufferele; graficele PYNQ 02/05 nu includ toate aceste etape.
Nu comparați aceste sume ca și cum ar avea aceleași limite.

Pentru verificare, comparați hash-ul intrării și hash-ul ieșirii cuantizate (în PYNQ folosiți
`conv_output` pentru 02 și `fpga_output` pentru 05):
```python
import hashlib
hashlib.sha256(np.ascontiguousarray(conv_output, dtype="<f4").tobytes()).hexdigest()
```
Rezultatele numerice ale etapelor float pot varia cu BLAS/CPU. Păstrați aceleași limite de threaduri,
faceți o încălzire și raportați mediana mai multor rulări. Notați temperatura, alimentarea și
serviciile active. uStreamer consumă resurse și trebuie menționat în rezultate.


In [ ]:
measurement = dict(input_name=input_name, input_sha256=input_sha256,
                   quantized_sha256=quantized_sha256, timings=dict(timings),
                   pipeline_sum_s=sum(timings.values()), machine=machine_info(),
                   background_camera_stream=BACKGROUND_CAMERA_STREAM)
for name, value in timings.items():
    print(f"{name}: {value:.4f} s")
print(f"Sum of measured stages: {measurement['pipeline_sum_s']:.4f} s")
print("Quantized output SHA256:", quantized_sha256)
PYNQ_QUANTIZED_CORE_S = None  # enter measured classifier.inference duration from PYNQ
if PYNQ_QUANTIZED_CORE_S is not None:
    if PYNQ_QUANTIZED_CORE_S <= 0:
        raise ValueError("PYNQ duration must be positive")
    print("CPU reference / FPGA call ratio:", timings['quantized_core_s'] / PYNQ_QUANTIZED_CORE_S)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(timings), list(timings.values()))
ax.invert_yaxis()
ax.set_xlabel("Measured seconds")
ax.set_title("Raspberry Pi CPU reference — one inference")
fig.tight_layout()
plt.show()
plt.close(fig)
# measurement stays in memory; no result JSON or capture file is saved.


## 6. Eliberare memorie

In [ ]:
classifier.deinit_accelerator()
darknet.close()
for name in ('conv0_weights', 'conv8_weights', 'conv0_bias', 'conv8_bias', 'original', 'preview'):
    globals().pop(name, None)
blas_limit.restore_original_limits()
gc.collect()
print('Models released; RAM temporary files have already been removed.')
